# Huấn luyện và so sánh model trên corpus legacy binary

Notebook này là giao diện thực thi theo từng bước cho `train_compare_legacy_binary_models.py`. Nó tái sử dụng đúng các hàm kiểm tra, feature và model của runner để tránh sai khác giữa notebook và CLI.

> **Phạm vi bắt buộc:** đây chỉ là nghiên cứu corpus legacy với mã nhãn `0/1`. Không diễn giải nhãn thành positive/negative, không triển khai model tạo ra vào API/dashboard và không dùng làm bảng kết quả chính cho sentiment cuộc họp.

## Chuẩn bị kernel

Mở notebook tại thư mục gốc repository và chọn kernel từ `.venv`. Nếu môi trường chưa có giao diện Jupyter, cài một lần trong virtual environment: `./.venv/bin/python -m pip install jupyterlab ipykernel`. Các package baseline cần có là `numpy`, `scikit-learn`, `joblib`. PhoBERT còn cần `torch`, `transformers`, `underthesea`, checkpoint đã cache/tải hợp lệ và GPU được khuyến nghị mạnh.

Chạy cell từ trên xuống dưới. Cell cấu hình từ chối thư mục output đã tồn tại để không ghi đè artifact cũ.

In [1]:
from __future__ import annotations

import json
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import sklearn


def find_project_root(start: Path) -> Path:
    marker = Path('Reports/New_folder/tram_cam/tools/train_compare_legacy_binary_models.py')
    for candidate in (start, *start.parents):
        if (candidate / marker).is_file():
            return candidate
    raise RuntimeError('Không tìm thấy project root. Hãy mở notebook từ trong repository.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
TOOLS_DIR = PROJECT_ROOT / 'Reports/New_folder/tram_cam/tools'
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

import train_compare_legacy_binary_models as legacy

CORPUS_DIR = PROJECT_ROOT / 'Reports/New_folder/tram_cam/derived/legacy-binary-tram-cam-v1'
RUN_NAME = 'legacy-binary-comparison-notebook-v4'  # Đổi tên khi cần chạy lại.
OUTPUT_DIR = CORPUS_DIR / 'experiments' / RUN_NAME
SEED = 42

# Giữ False khi chỉ chạy baseline. Bật True chỉ trên GPU và sau khi chuẩn bị checkpoint PhoBERT.
INCLUDE_PHOBERT = False
PHOBERT_CONFIG = {
    'model_name': 'vinai/phobert-base-v2',
    'revision': None,  # Có thể pin commit hash sau khi đã xác minh checkpoint.
    'max_length': 256,
    'epochs': 3,
    'learning_rate': 2e-5,
    'batch_size': 8,
    'device_preference': 'auto',
    'allow_cpu': False,
    'local_files_only': False,
}

print('Project root:', PROJECT_ROOT)
print('Corpus:', CORPUS_DIR)
print('Output mới:', OUTPUT_DIR)
print('Include PhoBERT:', INCLUDE_PHOBERT)

Project root: /home/lamsd/Documents/Buildkhoaluan
Corpus: /home/lamsd/Documents/Buildkhoaluan/Reports/New_folder/tram_cam/derived/legacy-binary-tram-cam-v1
Output mới: /home/lamsd/Documents/Buildkhoaluan/Reports/New_folder/tram_cam/derived/legacy-binary-tram-cam-v1/experiments/legacy-binary-comparison-notebook-v2


## Các hàm dùng trong notebook

Notebook gọi trực tiếp các hàm trong runner đã kiểm thử: `sha256_file`, `canonicalize`, `read_split`, `load_and_validate_corpus`, `length_features`, `build_models`, `model_input`, `evaluate`, `benchmark_single_request_latency`, `process_memory_snapshot`, `save_model_artifact`, `write_json` và `render_report`. `PhoBERTFineTuner` chỉ được tạo khi bật `INCLUDE_PHOBERT`. Do đó sửa một hàm trong runner sẽ được notebook dùng ở lần chạy kế tiếp.

In [2]:
FUNCTIONS = {
    name: getattr(legacy, name)
    for name in (
        'sha256_file', 'canonicalize', 'read_split', 'load_and_validate_corpus',
        'length_features', 'build_models', 'model_input', 'evaluate',
        'benchmark_single_request_latency', 'process_memory_snapshot',
        'memory_delta_megabytes', 'save_model_artifact', 'preflight_phobert',
        'write_json', 'render_report',
    )
}

for name, function in FUNCTIONS.items():
    first_line = (function.__doc__ or 'Không có docstring.').strip().splitlines()[0]
    print(f'- {name}: {first_line}')

- sha256_file: Không có docstring.
- canonicalize: Canonical form used only for integrity checks and model input cleanup.
- read_split: Không có docstring.
- load_and_validate_corpus: Không có docstring.
- length_features: Two non-semantic features: log character and whitespace-token counts.
- build_models: Return (id, display name, estimator, input kind), all fixed pre-registered specs.
- model_input: Không có docstring.
- evaluate: Không có docstring.
- write_json: Không có docstring.
- render_report: Không có docstring.


## 1. Kiểm tra corpus trước khi train

Cell này kiểm `dataset_kind`, SHA-256 của cả ba split, schema, nhãn `0/1`, duplicate trong split, leakage giữa split và độ khớp với `all_deduplicated.csv`. Nếu kiểm tra lỗi thì không train.

In [3]:
corpus, dataset_manifest = legacy.load_and_validate_corpus(CORPUS_DIR)
train_texts, train_labels = corpus['train']
validation_texts, validation_labels = corpus['validation']
test_texts, test_labels = corpus['test']

assert dataset_manifest['dataset_kind'] == 'legacy-binary-text-corpus'
assert set(train_labels + validation_labels + test_labels) == {0, 1}
print('Integrity checks: PASSED')
print('Dataset kind:', dataset_manifest['dataset_kind'])

Integrity checks: PASSED
Dataset kind: legacy-binary-text-corpus


In [4]:
split_rows = {split: len(corpus[split][0]) for split in legacy.SPLITS}
split_label_counts = {
    split: {str(label): int(Counter(corpus[split][1])[label]) for label in legacy.LABELS}
    for split in legacy.SPLITS
}
total_counts = Counter(train_labels + validation_labels + test_labels)

print(f"{'split':<12}{'rows':>8}{'code 0':>10}{'code 1':>10}")
for split in legacy.SPLITS:
    counts = split_label_counts[split]
    print(f"{split:<12}{split_rows[split]:>8}{counts['0']:>10}{counts['1']:>10}")
print(f"Total: {sum(split_rows.values())}; code-0:code-1 = {total_counts[0] / total_counts[1]:.3f}:1")

split           rows    code 0    code 1
train           7598      4572      3026
validation       949       571       378
test             949       571       378
Total: 9496; code-0:code-1 = 1.511:1


## 2. Các model được so sánh

Tất cả hyperparameter bên dưới là cấu hình cố định trước khi xem test. `length-only-logreg` chỉ là đối chứng để đo confound từ độ dài; nó không phải model ngữ nghĩa. Mỗi model được đo kích thước artifact, RSS process, GPU peak (nếu có) và latency local p50/p95 cho một request để sàng lọc kỹ thuật trước realtime. PhoBERT là ứng viên chất lượng tùy chọn: đặt `INCLUDE_PHOBERT = True` chỉ trên GPU. Không thêm model runtime của hệ thống cuộc họp vì contract nhãn của chúng không tương thích với mã `0/1` legacy.

In [5]:
PHOBERT_PREFLIGHT = None
if INCLUDE_PHOBERT:
    PHOBERT_PREFLIGHT = legacy.preflight_phobert(PHOBERT_CONFIG, SEED)
    print('PhoBERT preflight:', PHOBERT_PREFLIGHT)

MODEL_SPECS = legacy.build_models(
    SEED,
    include_phobert=INCLUDE_PHOBERT,
    phobert_config=PHOBERT_CONFIG,
)

for model_id, display_name, estimator, input_kind in MODEL_SPECS:
    print(f'- {model_id}: {display_name} | input={input_kind}')

# Baseline: majority-class, length-only-logreg, word-tfidf-logreg, char-tfidf-logreg.
# Ứng viên chất lượng (tùy chọn): phobert-finetune.

- majority-class: Majority-class baseline | input=placeholder_text
- length-only-logreg: Length-only Logistic Regression | input=length_features
- word-tfidf-logreg: Word TF-IDF (1–2 gram) + Logistic Regression | input=text
- char-tfidf-logreg: Character TF-IDF (3–5 gram) + Logistic Regression | input=text


## 3. Huấn luyện, đánh giá và lưu model

Mỗi model fit trên `train` duy nhất, đánh giá validation để xếp hạng và chạy test đúng một lần cho metric của cấu hình cố định. Sau đó notebook đo latency trên request đơn lẻ, RSS process, GPU peak (nếu dùng CUDA) và kích thước artifact, không dùng nhãn hay điều chỉnh hyperparameter. Cell từ chối ghi đè output. Baseline lưu `.joblib`; PhoBERT lưu thư mục Hugging Face với `safetensors`. Tất cả artifact chỉ giữ local.

In [6]:
if 'approved' in OUTPUT_DIR.resolve().parts:
    raise ValueError('Không được ghi experiment legacy vào approved/.')
if OUTPUT_DIR.exists():
    raise FileExistsError(f'Output đã tồn tại: {OUTPUT_DIR}. Đổi RUN_NAME trước khi chạy lại.')

MODELS_DIR = OUTPUT_DIR / 'models'
MODELS_DIR.mkdir(parents=True)
RUN_STARTED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
model_results = []

for model_id, display_name, estimator, input_kind in MODEL_SPECS:
    train_input = legacy.model_input(train_texts, input_kind)
    validation_input = legacy.model_input(validation_texts, input_kind)
    test_input = legacy.model_input(test_texts, input_kind)

    memory_before_fit = legacy.process_memory_snapshot()
    started = time.perf_counter()
    estimator.fit(train_input, train_labels)
    fit_seconds = time.perf_counter() - started
    memory_after_fit = legacy.process_memory_snapshot()

    validation_result = legacy.evaluate(estimator, validation_input, validation_labels)
    test_result = legacy.evaluate(estimator, test_input, test_labels)

    model_path = legacy.save_model_artifact(estimator, MODELS_DIR, model_id)
    single_request_latency = legacy.benchmark_single_request_latency(
        estimator, test_texts, input_kind
    )
    artifact_size_bytes = legacy.artifact_size_bytes(model_path)
    model_results.append({
        'model_id': model_id,
        'display_name': display_name,
        'input_kind': input_kind,
        'fit_seconds': round(fit_seconds, 6),
        'validation': validation_result,
        'test': test_result,
        'artifact_size_bytes': artifact_size_bytes,
        'artifact_size_megabytes': round(artifact_size_bytes / (1024 * 1024), 6),
        'single_request_latency': single_request_latency,
        'process_memory': {
            'before_fit': memory_before_fit,
            'after_fit': memory_after_fit,
            'rss_delta_megabytes': legacy.memory_delta_megabytes(memory_before_fit, memory_after_fit),
        },
        'runtime_metadata': getattr(estimator, 'runtime_metadata', {}),
        'artifact': str(model_path.relative_to(OUTPUT_DIR)),
    })

print(f'Đã train và lưu {len(model_results)} model vào: {MODELS_DIR}')

Đã train và lưu 4 model vào: /home/lamsd/Documents/Buildkhoaluan/Reports/New_folder/tram_cam/derived/legacy-binary-tram-cam-v1/experiments/legacy-binary-comparison-notebook-v2/models


## 4. Bảng so sánh chất lượng và benchmark kỹ thuật

Chọn theo validation Macro-F1, không chọn theo test. Bảng bổ sung kích thước artifact, RSS process delta, GPU peak (nếu có) và p50/p95 latency local của một request. Latency này không bao gồm model loading, transcript capture, network, API hay dashboard, nên chưa phải SLA realtime.

In [7]:
selected = max(
    model_results,
    key=lambda item: (item['validation']['macro_f1'], item['model_id']),
)

print(f"{'model':<26}{'val F1':>9}{'test F1':>9}{'MB':>8}{'RSS Δ':>9}{'GPU':>9}{'p50':>9}{'p95':>9}")
for item in model_results:
    print(
        f"{item['model_id']:<26}"
        f"{item['validation']['macro_f1']:>9.4f}"
        f"{item['test']['macro_f1']:>9.4f}"
        f"{item['artifact_size_megabytes']:>8.2f}"
        f"{(item['process_memory']['rss_delta_megabytes'] or 0):>9.2f}"
        f"{item['runtime_metadata'].get('cuda_peak_allocated_megabytes', 0):>9.2f}"
        f"{item['single_request_latency']['p50_milliseconds']:>9.3f}"
        f"{item['single_request_latency']['p95_milliseconds']:>9.3f}"
    )

print('\nỨng viên tạm thời:', selected['model_id'])
print('Validation Macro-F1:', f"{selected['validation']['macro_f1']:.4f}")

for item in model_results:
    print(f"\nConfusion matrix — {item['model_id']} (true rows, predicted columns; codes 0,1):")
    print(np.asarray(item['test']['confusion_matrix_true_rows_predicted_columns']))

model                                    val macro-F1   test macro-F1   test bal.acc.
majority-class                                 0.3757          0.3757          0.5000
length-only-logreg                             0.7722          0.7721          0.7831
word-tfidf-logreg                              0.9857          0.9868          0.9864
char-tfidf-logreg                              0.9901          0.9868          0.9868

Ứng viên tạm thời: char-tfidf-logreg
Validation Macro-F1: 0.9901

Confusion matrix — majority-class (true rows, predicted columns; codes 0,1):
[[571   0]
 [378   0]]

Confusion matrix — length-only-logreg (true rows, predicted columns; codes 0,1):
[[426 145]
 [ 68 310]]

Confusion matrix — word-tfidf-logreg (true rows, predicted columns; codes 0,1):
[[566   5]
 [  7 371]]

Confusion matrix — char-tfidf-logreg (true rows, predicted columns; codes 0,1):
[[565   6]
 [  6 372]]


## 5. Ghi artifact tái lập

Cell này tạo `results.json`, `run-manifest.json` và `comparison.md`. Các file chỉ có aggregate metrics, cấu hình, hash, artifact size, RSS/GPU memory và latency local; không ghi mẫu text thô vào report/log.

In [8]:
results = {
    'scope': {
        'warning': legacy.SCOPE_WARNING,
        'allowed_use': 'local legacy research comparison only',
        'excluded_from': [
            'meeting-sentiment production runtime', 'API and dashboard',
            'source/model-artifacts', 'source/model-data/approved',
            'Chapter 8 primary meeting-sentiment results',
        ],
    },
    'dataset': {
        'name': dataset_manifest.get('dataset_name', CORPUS_DIR.name),
        'dataset_kind': dataset_manifest['dataset_kind'],
        'corpus_dir': str(CORPUS_DIR),
        'output_sha256_verified': {
            split: legacy.sha256_file(CORPUS_DIR / f'{split}.csv')
            for split in legacy.SPLITS
        },
        'manifest_sha256': legacy.sha256_file(CORPUS_DIR / 'dataset-manifest.json'),
    },
    'run': {
        'started_at_utc': RUN_STARTED_AT,
        'finished_at_utc': datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        'seed': SEED,
        'python': sys.version.split()[0],
        'numpy': np.__version__,
        'scikit_learn': sklearn.__version__,
        'joblib': joblib.__version__,
    },
    'corpus': {
        'split_rows': split_rows,
        'split_label_counts': split_label_counts,
        'total_rows': sum(split_rows.values()),
        'total_label_counts': {str(label): int(total_counts[label]) for label in legacy.LABELS},
        'class_ratio_code_0_to_code_1': total_counts[0] / total_counts[1],
        'split_policy': 'Train only for fit; validation ranks fixed models; test is evaluated once per fixed model.',
    },
    'fixed_model_specs': {
        'include_phobert': INCLUDE_PHOBERT,
        'phobert_config': PHOBERT_CONFIG if INCLUDE_PHOBERT else None,
        'phobert_environment_preflight': PHOBERT_PREFLIGHT,
    },
    'models': model_results,
    'provisional_selection': {
        'model_id': selected['model_id'],
        'display_name': selected['display_name'],
        'validation_macro_f1': selected['validation']['macro_f1'],
        'selection_rule': 'highest validation macro-F1; fixed configurations only',
    },
    'realtime_readiness': {
        'status': 'not_eligible_for_meeting_realtime_from_legacy_corpus',
        'technical_screening_available': True,
        'phobert_included': INCLUDE_PHOBERT,
        'local_benchmark_scope': 'Single-request local latency includes model_input and predict only; it excludes I/O and is not an SLA.',
        'required_before_any_realtime_selection': [
            'approved meeting transcript corpus with consent',
            'negative/neutral/positive/unknown labels and meeting_id split',
            'end-to-end latency in the deployment environment',
            'shadow mode with human review',
        ],
    },
    'limitations': [
        'Label codes 0/1 have no assigned meeting-sentiment meaning.',
        'No meeting_id exists, so meeting-level holdout and bootstrap CI are unavailable.',
        'EDA found provenance/label and length confounds; metrics are not proof of semantic understanding.',
    ],
}

legacy.write_json(OUTPUT_DIR / 'results.json', results)
legacy.write_json(
    OUTPUT_DIR / 'run-manifest.json',
    {
        'scope_warning': legacy.SCOPE_WARNING,
        'execution': 'Run all cells in legacy_binary_model_comparison.ipynb from top to bottom.',
        'input_manifest': str(CORPUS_DIR / 'dataset-manifest.json'),
        'verified_split_hashes': results['dataset']['output_sha256_verified'],
        'fixed_models': [item['model_id'] for item in model_results],
        'include_phobert': INCLUDE_PHOBERT,
        'phobert_config': PHOBERT_CONFIG if INCLUDE_PHOBERT else None,
        'phobert_environment_preflight': PHOBERT_PREFLIGHT,
        'no_test_tuning': True,
        'single_request_latency_policy': '50 evenly spaced test records after 10 warm-up requests; local model_input and predict only.',
    },
)
(OUTPUT_DIR / 'comparison.md').write_text(legacy.render_report(results), encoding='utf-8')

print('Artifacts created:')
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print('-', path.relative_to(OUTPUT_DIR))

Artifacts created:
- comparison.md
- models/char-tfidf-logreg.joblib
- models/length-only-logreg.joblib
- models/majority-class.joblib
- models/word-tfidf-logreg.joblib
- results.json
- run-manifest.json


## Diễn giải đúng phạm vi

So sánh này chỉ trả lời model nào tái tạo tốt nhất mã nhãn legacy trong split hiện có. Nếu `length-only-logreg` có điểm đáng kể hoặc TF-IDF/PhoBERT đạt điểm rất cao, xem đó là tín hiệu cần kiểm tra confound về độ dài/provenance, không phải bằng chứng model hiểu cảm xúc hay có thể dùng trong Google Meet/Vexa. PhoBERT chỉ là ứng viên chất lượng để benchmark: trước deploy cần corpus transcript đã phê duyệt, ba nhãn, `meeting_id`, split theo cuộc họp, đo latency end-to-end, RAM GPU/CPU mục tiêu và shadow mode có người review.